# Experiment H: Ensemble — Supervised RF + Unsupervised Cluster Regions

Two independent systems vote on each cycle:
- **System A (Supervised):** Exp G's RF on raw 768D → predicts label + confidence
- **System B (Unsupervised):** R6 signed axis K-means → assigns to a region → region's dominant label

Rules:
- Both agree → use that label (high confidence)
- Disagree → trust the one with stronger evidence, or reject as uncertain

In [ ]:
!git clone https://github.com/helenejabbour/ropeflow-project.git 2>/dev/null || echo 'Already cloned'
import os
BASE = 'ropeflow-project'
DATA_PROCESSED = os.path.join(BASE, 'data', 'processed')
NEW_LABELED_RAW = os.path.join(BASE, 'data', 'raw', 'new-labeled-sessions')
print('Setup done')

In [ ]:
import glob, re, json as _json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import savgol_filter, find_peaks
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, classification_report, confusion_matrix
from collections import Counter, defaultdict
import warnings
warnings.filterwarnings('ignore')
np.random.seed(42)
print('Imports OK')

In [ ]:
CONFIG = {
    'FS': 50.0, 'CYCLE_PROMINENCE_DEGS': 100.0,
    'CYCLE_SAVGOL_WINDOW': 21, 'CYCLE_SAVGOL_POLYORDER': 3,
    'CYCLE_MIN_PEAK_DEGS': 300.0, 'CYCLE_MIN_PERIOD_S': 0.5,
    'CYCLE_MAX_PERIOD_S': 3.0, 'TARGET_LEN': 64, 'MIN_CYCLE_SAMPLES': 10,
}
KNOWN_PATTERNS = {'overhand','sneak_overhand','underhand','sneak_underhand','dragon_roll','underhand_default'}
LR_CLASSES = {'UR','UL','OR','OL','USR','USL','OSR','OSL','FB','BF'}
COARSE_MAP = {'UR':'underhand','UL':'underhand','OR':'overhand','OL':'overhand',
    'USR':'sneak_underhand','USL':'sneak_underhand','OSR':'sneak_overhand','OSL':'sneak_overhand',
    'FB':'dragon_roll','BF':'dragon_roll'}
GENERIC_TO_LR = {'U_generic':['UR','UL'],'O_generic':['OR','OL'],
    'DR_generic':['FB','BF'],'SU_generic':['USR','USL'],'SO_generic':['OSR','OSL']}
FINE_MAP = {'UR':'UR','ur':'UR','UR0':'UR','UR-CW':'UR','UL0':'UL',
    'OR':'OR','or':'OR','OR2':'OR','OR3':'OR','OR-OSL':'OR','OR/OSL?':'OR',
    'OL':'OL','ol':'OL','OL2':'OL','USR':'USR','USL':'USL','OSR':'OSR','OSL':'OSL',
    'FB':'FB','fb':'FB','FB2':'FB','BF2':'BF','bf':'BF',
    'underhand':'U_generic','overhand':'O_generic','dragon_roll':'DR_generic',
    'sneak_underhand':'SU_generic','sneak_overhand':'SO_generic',
    'CW':None,'cw':None,'CCW':None,'ccw':None,'idle':None,'Idle3':None,'Idle-or-ol?':None,'VQ5':None,'vq16':None}
_FP = [(re.compile(r'^USR$',re.I),'USR'),(re.compile(r'^USL$',re.I),'USL'),
    (re.compile(r'^OSR$',re.I),'OSR'),(re.compile(r'^OSL$',re.I),'OSL'),
    (re.compile(r'^UR',re.I),'UR'),(re.compile(r'^UL',re.I),'UL'),
    (re.compile(r'^OR',re.I),'OR'),(re.compile(r'^OL',re.I),'OL'),
    (re.compile(r'^FB',re.I),'FB'),(re.compile(r'^BF',re.I),'BF'),
    (re.compile(r'^CW$',re.I),None),(re.compile(r'^CCW$',re.I),None),
    (re.compile(r'^idle',re.I),None),(re.compile(r'^vq',re.I),None)]
def map_fine(raw):
    raw=raw.strip()
    if raw in FINE_MAP: return FINE_MAP[raw]
    if raw.lower() in FINE_MAP: return FINE_MAP[raw.lower()]
    for p,c in _FP:
        if p.match(raw): return c
    return None
def extract_signals(df):
    t=df['timestamp_ms'].values/1000.0; A=df[['ax_w','ay_w','az_w']].values
    om=df[['gx','gy','gz']].values*(np.pi/180.0); return t,A,om
def detect_cycles(t,om,fs=50.0):
    md=np.linalg.norm(om,axis=1)*(180/np.pi); n=len(md)
    if n<7: return []
    w=int(CONFIG['CYCLE_SAVGOL_WINDOW'])
    if w%2==0: w+=1
    w=max(5,min(w,n if n%2==1 else n-1)); po=max(1,min(int(CONFIG['CYCLE_SAVGOL_POLYORDER']),w-2))
    ms=savgol_filter(md,window_length=w,polyorder=po,mode='interp')
    ms=savgol_filter(ms,window_length=w,polyorder=po,mode='interp')
    pks,_=find_peaks(ms,distance=max(1,int(CONFIG['CYCLE_MIN_PERIOD_S']*fs)),
        prominence=CONFIG['CYCLE_PROMINENCE_DEGS'],height=CONFIG['CYCLE_MIN_PEAK_DEGS'])
    if len(pks)==0: return []
    cyc=[]
    for i,p in enumerate(pks):
        l=0 if i==0 else (pks[i-1]+p)//2; r=(len(t)-1) if i==len(pks)-1 else (p+pks[i+1])//2
        if r>l and (r-l)>=CONFIG['MIN_CYCLE_SAMPLES']: cyc.append((l,r))
    return cyc
def pair_cycles(t0,c0,t1,c1):
    p0,p1,u=[],[],set()
    for a in c0:
        bi,bo=-1,-1.0
        for i,b in enumerate(c1):
            if i in u: continue
            o=max(0,min(t0[a[1]],t1[b[1]])-max(t0[a[0]],t1[b[0]]))
            if o>bo: bo,bi=o,i
        if bi>=0 and bo>0: p0.append(a);p1.append(c1[bi]);u.add(bi)
    return p0,p1
def resample(s,n):
    if len(s)<2: return np.zeros(n) if s.ndim==1 else np.zeros((n,s.shape[1]))
    xo,xn=np.linspace(0,1,len(s)),np.linspace(0,1,n)
    if s.ndim==1: return np.interp(xn,xo,s)
    return np.column_stack([np.interp(xn,xo,s[:,j]) for j in range(s.shape[1])])
def build_cm(A0,A1,om0,om1,s0,e0,s1,e1):
    tl=CONFIG['TARGET_LEN']
    r0=resample(np.column_stack([A0[s0:e0],om0[s0:e0]]),tl)
    r1=resample(np.column_stack([A1[s1:e1],om1[s1:e1]]),tl)
    return np.column_stack([r0,r1]).T.astype(np.float32)
def signed_axis_features(cm):
    feats=[]
    for ch in [3,4,5,9,10,11]:
        feats.append(np.mean(cm[ch])); feats.append(np.std(cm[ch])); feats.append(np.mean(np.sign(cm[ch])))
    return np.array(feats,dtype=np.float32)
print('Functions defined')

In [ ]:
# ── Load all labeled cycles ───────────────────────────────────
all_cms=[]; all_fine=[]; all_groups=[]
def load_c(d0p,d1p,name,fine='unlabeled',windows=None):
    df0,df1=pd.read_csv(d0p),pd.read_csv(d1p)
    t0,A0,om0=extract_signals(df0); t1,A1,om1=extract_signals(df1)
    c0=detect_cycles(t0,om0,CONFIG['FS']); c1=detect_cycles(t1,om1,CONFIG['FS'])
    p0,p1=pair_cycles(t0,c0,t1,c1)
    if windows:
        fp0,fp1=[],[]
        for (s0,e0),(s1,e1) in zip(p0,p1):
            if any(ws<=(t0[s0]+t0[e0])/2<we for ws,we in windows): fp0.append((s0,e0));fp1.append((s1,e1))
        p0,p1=fp0,fp1
    g=name.split('/')[0] if '/' in name else name; cnt=0
    for (s0,e0),(s1,e1) in zip(p0,p1):
        all_cms.append(build_cm(A0,A1,om0,om1,s0,e0,s1,e1))
        all_fine.append(fine); all_groups.append(g); cnt+=1
    return cnt
for d0 in sorted(glob.glob(os.path.join(DATA_PROCESSED,'*_device0_processed.csv'))):
    d1=d0.replace('_device0_','_device1_')
    if not os.path.exists(d1): continue
    stem=os.path.basename(d0).replace('_device0_processed.csv',''); parts=stem.split('_')
    if len(parts)<3: continue
    rest='_'.join(parts[2:]); fine='unlabeled'
    for pat in sorted(KNOWN_PATTERNS,key=len,reverse=True):
        if rest.startswith(pat):
            if pat in('underhand','underhand_default'): fine='U_generic'
            elif pat=='overhand': fine='O_generic'
            elif pat=='dragon_roll': fine='DR_generic'
            elif pat=='sneak_underhand': fine='SU_generic'
            elif pat=='sneak_overhand': fine='SO_generic'
            break
    if fine=='unlabeled': continue
    load_c(d0,d1,stem,fine)
if os.path.isdir(NEW_LABELED_RAW):
    for sn in sorted(os.listdir(NEW_LABELED_RAW)):
        sd=os.path.join(NEW_LABELED_RAW,sn)
        if not os.path.isdir(sd): continue
        lp=None
        for fn in('labels_corrected.json','labels.json'):
            p=os.path.join(sd,fn)
            if os.path.isfile(p): lp=p; break
        if not lp: continue
        d0=os.path.join(DATA_PROCESSED,sn+'_device0_processed.csv')
        d1=os.path.join(DATA_PROCESSED,sn+'_device1_processed.csv')
        if not(os.path.isfile(d0) and os.path.isfile(d1)): continue
        with open(lp,encoding='utf-8') as f: data=_json.load(f)
        segs=data.get('segments',data) if isinstance(data,dict) else data
        wbl=defaultdict(list)
        for seg in segs:
            fi=map_fine(seg.get('label',''))
            if fi is None: continue
            s,e=seg.get('start'),seg.get('end')
            if s is None: continue
            if e is None: e=s+2.0
            wbl[fi].append((float(s),float(e)))
        for fi,wins in sorted(wbl.items()): load_c(d0,d1,sn+'/'+fi,fi,wins)

CMS = np.array(all_cms); y_fine = np.array(all_fine); g_all = np.array(all_groups)
X_raw = CMS.reshape(len(CMS), -1)
X_signed = np.array([signed_axis_features(cm) for cm in CMS])
lr_mask = np.array([l in LR_CLASSES for l in y_fine])
gen_mask = np.array([l in GENERIC_TO_LR for l in y_fine])
print(f'L/R: {lr_mask.sum()}, Generic: {gen_mask.sum()}, Total: {len(CMS)}')

In [ ]:
# ── Expand generic labels (Exp G, thresh=0.5) ────────────────
X_lr=X_raw[lr_mask]; y_lr=y_fine[lr_mask]; g_lr=g_all[lr_mask]
X_gen=X_raw[gen_mask]; y_gen=y_fine[gen_mask]; g_gen=g_all[gen_mask]
X_signed_gen=X_signed[gen_mask]

sc_tmp=StandardScaler(); rf_tmp=RandomForestClassifier(n_estimators=400,class_weight='balanced',random_state=42)
rf_tmp.fit(sc_tmp.fit_transform(X_lr), y_lr)
pred_gen=rf_tmp.predict(sc_tmp.transform(X_gen))
pred_conf=np.max(rf_tmp.predict_proba(sc_tmp.transform(X_gen)),axis=1)
y_exp=[]
for i in range(len(X_gen)):
    allowed=set(GENERIC_TO_LR.get(y_gen[i],[]))
    if pred_gen[i] in allowed: y_exp.append(pred_gen[i])
    else:
        probs={c:rf_tmp.predict_proba(sc_tmp.transform(X_gen[i:i+1]))[0,list(rf_tmp.classes_).index(c)]
               for c in allowed if c in rf_tmp.classes_}
        y_exp.append(max(probs,key=probs.get) if probs else 'unknown')
y_exp=np.array(y_exp)
valid_exp=(y_exp!='unknown')&(pred_conf>=0.5)
X_gen_exp=X_gen[valid_exp]; y_gen_exp=y_exp[valid_exp]; g_gen_exp=g_gen[valid_exp]
X_signed_gen_exp=X_signed_gen[valid_exp]
print(f'Expanded: {valid_exp.sum()} generic cycles at thresh=0.5')

In [ ]:
# ── LOSO with ensemble: RF prediction + cluster region vote ──

unique_groups = np.unique(g_lr)
class_groups = defaultdict(set)
for l,g in zip(y_lr,g_lr): class_groups[l].add(g)
singleton = {c for c,gs in class_groups.items() if len(gs)<2}
test_groups = [g for g in unique_groups if not set(y_lr[g_lr==g]).issubset(singleton)]

# Run 4 strategies:
strategies = {
    'RF only (Exp G baseline)': {},
    'Cluster only (nearest region)': {},
    'Ensemble: agree or RF': {},
    'Ensemble: agree or reject': {},
}

for strat in strategies:
    strategies[strat] = {'at10':[],'ap10':[],'at5':[],'ap5':[],'n_agree':0,'n_disagree':0,'n_total':0}

print('Running LOSO ensemble...')
for gi, g in enumerate(test_groups):
    te = g_lr == g; tr = ~te
    # Training data: L/R train + expanded generic (not in test group)
    gen_tr = g_gen_exp != g
    X_tr_raw = np.vstack([X_lr[tr], X_gen_exp[gen_tr]])
    y_tr = np.concatenate([y_lr[tr], y_gen_exp[gen_tr]])
    X_tr_signed = np.vstack([X_signed[lr_mask][tr], X_signed_gen_exp[gen_tr]])
    
    X_te_raw = X_lr[te]; y_te = y_lr[te]
    X_te_signed = X_signed[lr_mask][te]
    
    # System A: RF on raw 768D
    sc_a = StandardScaler(); X_tr_a = sc_a.fit_transform(X_tr_raw); X_te_a = sc_a.transform(X_te_raw)
    rf = RandomForestClassifier(n_estimators=400, class_weight='balanced', random_state=42)
    rf.fit(X_tr_a, y_tr)
    rf_pred = rf.predict(X_te_a)
    rf_conf = np.max(rf.predict_proba(X_te_a), axis=1)
    
    # System B: K-means on signed axis 18D
    sc_b = StandardScaler(); X_tr_b = sc_b.fit_transform(X_tr_signed); X_te_b = sc_b.transform(X_te_signed)
    km = KMeans(n_clusters=10, random_state=42, n_init=20, max_iter=500)
    km.fit(X_tr_b)
    # Map clusters to labels
    tr_clusters = km.labels_
    cluster_map = {}
    for c in range(10):
        cmask = tr_clusters == c
        if cmask.sum() == 0: cluster_map[c] = 'unknown'; continue
        cluster_map[c] = Counter(y_tr[cmask]).most_common(1)[0][0]
    te_clusters = km.predict(X_te_b)
    cluster_pred = np.array([cluster_map.get(c, 'unknown') for c in te_clusters])
    
    # Evaluate each strategy
    for i in range(len(y_te)):
        rf_p = rf_pred[i]; cl_p = cluster_pred[i]; true = y_te[i]
        agree = (rf_p == cl_p)
        
        for strat, d in strategies.items():
            d['n_total'] += 1
            if agree: d['n_agree'] += 1
            else: d['n_disagree'] += 1
            
            if strat == 'RF only (Exp G baseline)':
                final = rf_p
            elif strat == 'Cluster only (nearest region)':
                final = cl_p
            elif strat == 'Ensemble: agree or RF':
                final = rf_p  # when they agree, same answer; when disagree, trust RF
            elif strat == 'Ensemble: agree or reject':
                if agree:
                    final = rf_p
                else:
                    continue  # skip this cycle — no prediction
            
            if final == 'unknown': continue
            d['at10'].append(true); d['ap10'].append(final)
            d['at5'].append(COARSE_MAP[true]); d['ap5'].append(COARSE_MAP.get(final, final))
    
    if (gi+1) % 3 == 0: print(f'  fold {gi+1}/{len(test_groups)}')

print('Done')

In [ ]:
# ── Compute metrics for each strategy ─────────────────────────

results = []
for strat, d in strategies.items():
    at10=np.array(d['at10']); ap10=np.array(d['ap10'])
    at5=np.array(d['at5']); ap5=np.array(d['ap5'])
    if len(at10) == 0:
        results.append({'name':strat,'f1_10':0,'f1_5':0,'acc':0,'coverage':0})
        continue
    labs10=sorted(set(at10)|set(ap10)); labs5=sorted(set(at5)|set(ap5))
    f1_10=f1_score(at10,ap10,average='macro',labels=labs10,zero_division=0)
    f1_5=f1_score(at5,ap5,average='macro',labels=labs5,zero_division=0)
    acc=np.mean(at10==ap10)
    coverage=len(at10)/d['n_total'] if d['n_total']>0 else 0
    agree_rate=d['n_agree']/d['n_total'] if d['n_total']>0 else 0
    results.append({'name':strat,'f1_10':f1_10,'f1_5':f1_5,'acc':acc,
                    'coverage':coverage,'agree_rate':agree_rate,
                    'n_predicted':len(at10),'n_total':d['n_total'],
                    'cm10':confusion_matrix(at10,ap10,labels=labs10),'labs10':labs10,
                    'cm5':confusion_matrix(at5,ap5,labels=labs5),'labs5':labs5,
                    'report':classification_report(at10,ap10,labels=labs10,zero_division=0)})
    print(f'{strat}')
    print(f'  10-class F1={f1_10:.3f}, 5-class F1={f1_5:.3f}, Acc={acc:.3f}')
    print(f'  Coverage={coverage:.1%} ({len(at10)}/{d["n_total"]}), Agree rate={agree_rate:.1%}')

In [ ]:
# ── Plots ─────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

names = [r['name'] for r in results]
f5s = [r['f1_5'] for r in results]
f10s = [r['f1_10'] for r in results]
covs = [r.get('coverage',1) for r in results]

x = np.arange(len(names))
axes[0].barh(x-0.15, f5s, 0.3, color='#5DCAA5', label='5-class F1')
axes[0].barh(x+0.15, f10s, 0.3, color='#7F77DD', label='10-class F1')
axes[0].set_yticks(x); axes[0].set_yticklabels(names, fontsize=8)
axes[0].axvline(0.823, color='gray', ls='--', lw=1, label='Exp G (0.823)')
axes[0].set_xlabel('LOSO F1'); axes[0].legend(fontsize=7); axes[0].set_title('F1 Scores')

axes[1].barh(x, [c*100 for c in covs], color='#E8A838')
axes[1].set_yticks(x); axes[1].set_yticklabels(names, fontsize=8)
axes[1].set_xlabel('Coverage %'); axes[1].set_title('Coverage (% of cycles predicted)')
plt.tight_layout(); plt.show()

# Best strategy confusion matrix
best = max(results, key=lambda r: r['f1_5'])
if 'cm5' in best:
    fig, ax = plt.subplots(figsize=(8, 6))
    im = ax.imshow(best['cm5'], cmap='Blues')
    ax.set_xticks(range(len(best['labs5']))); ax.set_yticks(range(len(best['labs5'])))
    ax.set_xticklabels(best['labs5'], rotation=45, ha='right')
    ax.set_yticklabels(best['labs5'])
    for i in range(len(best['labs5'])):
        for j in range(len(best['labs5'])):
            ax.text(j,i,str(best['cm5'][i,j]),ha='center',va='center',
                    color='white' if best['cm5'][i,j]>best['cm5'].max()*0.5 else 'black')
    ax.set_title(best['name'] + f' (F1={best["f1_5"]:.3f})'); plt.colorbar(im, ax=ax)
    plt.tight_layout(); plt.show()

In [ ]:
# ── SUMMARY ──────────────────────────────────────────────────
print('='*70)
print('EXPERIMENT H: ENSEMBLE SUP + UNSUP SUMMARY')
print('='*70)
print(f'System A: RF on raw 768D (Exp G with expansion)')
print(f'System B: K-means k=10 on signed axis 18D (R6)')
print(f'')
print(f'{"Strategy":<40s}  {"F1-10":>6s}  {"F1-5":>6s}  {"Acc":>6s}  {"Cover":>6s}')
print('-'*70)
for r in sorted(results, key=lambda r: r['f1_5'], reverse=True):
    cov = f'{r.get("coverage",1):.0%}' if r.get('coverage',1) < 1 else '100%'
    print(f'{r["name"]:<40s}  {r["f1_10"]:>6.3f}  {r["f1_5"]:>6.3f}  {r["acc"]:>6.3f}  {cov:>6s}')
print(f'')
agree_r = results[2] if len(results)>2 else results[0]  # 'agree or RF'
print(f'Agreement rate (RF and cluster agree): {agree_r.get("agree_rate",0):.1%}')
print(f'')
print(f'References:')
print(f'  V08:    F1=0.632')
print(f'  E1:     F1=0.811')
print(f'  Exp G:  F1=0.823')
best = max(results, key=lambda r: r['f1_5'])
print(f'  Best H: F1={best["f1_5"]:.3f} ({best["name"]})')